# 02 Missing Data & Feature Engineering

Loads the raw CSVs saved in notebook 1, engineers event-level features 
from the raw columns, aggregates to driver-season level, identifies missing 
values, and applies imputation to produce a clean processed dataset.

**Input:** `data/raw/race_results_2021_2025.csv`, `data/raw/qualifying_results_2021_2025.csv`  
**Output:** `data/processed/driver_season_summary.csv`

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [13]:
RAW_DIR       = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

race_df  = pd.read_csv(RAW_DIR / 'race_results_2021_2025.csv')
quali_df = pd.read_csv(RAW_DIR / 'qualifying_results_2021_2025.csv')

print("Race data:  ", race_df.shape)
print("Quali data: ", quali_df.shape)
print("\nRace columns:", list(race_df.columns))

Race data:   (2278, 12)
Quali data:  (2277, 12)

Race columns: ['season', 'round', 'race_name', 'date', 'driver_id', 'driver_code', 'driver_name', 'constructor', 'grid', 'finish_position', 'points', 'status']


## Feature Engineering


In [14]:
race_df['is_win']    = (race_df['finish_position'] == 1).astype(int)
race_df['is_podium'] = race_df['finish_position'].isin([1, 2, 3]).astype(int)
race_df['is_points'] = (race_df['finish_position'] <= 10).astype(int)
race_df['is_dnf']    = (~race_df['status'].str.contains(
                            'Finished', case=False, na=False)).astype(int)
race_df['positions_gained'] = race_df['grid'] - race_df['finish_position']

print("Features added. Sample:")
race_df[['driver_name', 'season', 'round', 'grid',
         'finish_position', 'is_win', 'is_podium',
         'is_dnf', 'positions_gained']].head(8)

Features added. Sample:


,driver_name,season,round,grid,finish_position,is_win,is_podium,is_dnf,positions_gained
0,Lewis Hamilton,2021,1,2,1,1,1,0,1
1,Max Verstappen,2021,1,1,2,0,1,0,-1
2,Valtteri Bottas,2021,1,3,3,0,1,0,0
3,Lando Norris,2021,1,7,4,0,0,0,3
4,Sergio Pérez,2021,1,0,5,0,0,0,-5
5,Charles Leclerc,2021,1,4,6,0,0,0,-2
6,Daniel Ricciardo,2021,1,6,7,0,0,0,-1
7,Carlos Sainz,2021,1,8,8,0,0,0,0


## Aggregation to Driver-Season Level


In [15]:
race_summary = (
    race_df
    .groupby(['season', 'driver_id', 'driver_code', 'driver_name', 'constructor'])
    .agg(
        races                = ('round',            'count'),
        total_points         = ('points',           'sum'),
        points_per_race      = ('points',           'mean'),
        avg_finish_position  = ('finish_position',  'mean'),
        finish_position_std  = ('finish_position',  'std'),
        wins                 = ('is_win',           'sum'),
        podiums              = ('is_podium',        'sum'),
        points_finishes      = ('is_points',        'sum'),
        dnfs                 = ('is_dnf',           'sum'),
        dnf_rate             = ('is_dnf',           'mean'),
        avg_positions_gained = ('positions_gained', 'mean')
    )
    .reset_index()
)

race_summary['points_finish_rate'] = race_summary['points_finishes'] / race_summary['races']
race_summary['podium_rate']        = race_summary['podiums']         / race_summary['races']
race_summary['win_rate']           = race_summary['wins']            / race_summary['races']

print("Before filter:", race_summary.shape)
print("\nDrivers with fewer than 3 races:")
print(race_summary[race_summary['races'] < 3][['driver_name', 'season', 'races']].to_string(index=False))

Before filter: (113, 19)

Drivers with fewer than 3 races:
    driver_name  season  races
  Robert Kubica    2021      2
  Nyck de Vries    2022      1
Nico Hülkenberg    2022      2
 Oliver Bearman    2024      1
 Oliver Bearman    2024      2
    Jack Doohan    2024      1
    Liam Lawson    2025      2
   Yuki Tsunoda    2025      2


In [16]:
race_summary = race_summary[race_summary['races'] >= 3].reset_index(drop=True)
print("After filter:", race_summary.shape)

After filter: (105, 19)


## Qualifying Data Merge

In [17]:
quali_summary = (
    quali_df
    .groupby(['season', 'driver_id'])
    .agg(avg_quali_position=('quali_position', 'mean'))
    .reset_index()
)

driver_season_df = race_summary.merge(
    quali_summary[['season', 'driver_id', 'avg_quali_position']],
    on=['season', 'driver_id'],
    how='left'
)

print("Combined shape:", driver_season_df.shape)
print("\nMissing values per column:")
print(driver_season_df.isna().sum()[driver_season_df.isna().sum() > 0])

Combined shape: (105, 20)

Missing values per column:
Series([], dtype: int64)


In [18]:
print("Drivers with missing finish_position_std:")
mask = driver_season_df['finish_position_std'].isna()
print(driver_season_df[mask][['driver_name', 'season', 'races']].to_string(index=False))

Drivers with missing finish_position_std:
Empty DataFrame
Columns: [driver_name, season, races]
Index: []


## Save Processed Dataset

In [19]:
driver_season_df.to_csv(
    PROCESSED_DIR / 'driver_season_summary.csv', index=False
)

print("Saved: data/processed/driver_season_summary.csv")
print("Shape:", driver_season_df.shape)
print("\nSample — top 5 by total points:")
print(
    driver_season_df
    .sort_values('total_points', ascending=False)
    [['driver_name', 'season', 'races', 'total_points',
      'wins', 'podiums', 'dnf_rate', 'avg_quali_position']]
    .head()
    .to_string(index=False)
)

Saved: data/processed/driver_season_summary.csv
Shape: (105, 20)

Sample — top 5 by total points:
   driver_name  season  races  total_points  wins  podiums  dnf_rate  avg_quali_position
Max Verstappen    2023     22         530.0    19       21  0.000000            3.000000
Max Verstappen    2022     22         433.0    15       17  0.090909            2.590909
Max Verstappen    2024     24         399.0     9       14  0.041667            2.916667
  Lando Norris    2025     24         394.0     7       18  0.125000            2.958333
Max Verstappen    2025     24         389.0     8       15  0.041667            3.500000


## Feature Selection


In [20]:
cols_to_drop = ['win_rate', 'podium_rate', 'points_finishes']
driver_season_df = driver_season_df.drop(columns=cols_to_drop)

print("Remaining columns:")
print(list(driver_season_df.columns))
print("\nFinal shape:", driver_season_df.shape)

Remaining columns:
['season', 'driver_id', 'driver_code', 'driver_name', 'constructor', 'races', 'total_points', 'points_per_race', 'avg_finish_position', 'finish_position_std', 'wins', 'podiums', 'dnfs', 'dnf_rate', 'avg_positions_gained', 'points_finish_rate', 'avg_quali_position']

Final shape: (105, 17)


Before saving, redundant features are removed from the index feature set.
Win rate and podium rate are dropped because they are taken directly from
wins and podiums divided by races, including both would double-count the
same information in the index. The raw counts are kept as they better
distinguish elite performances.

In [21]:
driver_season_df.to_csv(
    PROCESSED_DIR / 'driver_season_summary.csv', index=False
)
print("Saved clean dataset: data/processed/driver_season_summary.csv")
print("Shape:", driver_season_df.shape)

Saved clean dataset: data/processed/driver_season_summary.csv
Shape: (105, 17)
